# 環境確認

uv で構築した環境のパッケージバージョンと動作を確認します。

## 実行環境の選択

右上のカーネル選択で環境を切り替えます。`ENV = "auto"` のままで自動判定されます。

| カーネル選択 | ENV の判定 | 動作 |
|---|---|---|
| `ezkl (.venv)` | `local` | pip install スキップ（uv 環境を使用） |
| Colab ランタイム | `colab` | pip install を自動実行 |

> **VS Code + Colab 拡張機能の場合**  
> 右上のカーネル選択 → `Select Another Kernel` → `Existing Jupyter Server` → Colab ランタイムを選択

In [ ]:
# ── 実行環境 ──────────────────────────────────────────
# "auto"  : 自動検出（推奨）
# "colab" : Google Colab で強制実行
# "local" : ローカル (uv) で強制実行
ENV = "auto"
# ──────────────────────────────────────────────────────

if ENV == "auto":
    try:
        import google.colab
        ENV = "colab"
    except ImportError:
        ENV = "local"

print(f"実行環境: {ENV}")

if ENV == "colab":
    import subprocess, sys
    pkgs = ["ezkl", "onnx", "torch", "torchvision", "matplotlib", "pandas"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    print("パッケージのインストール完了")
else:
    print("ローカル環境 (uv) を使用します — インストール不要")

## Python バージョン

In [ ]:
import sys
print(f"Python: {sys.version}")
print(f"実行パス: {sys.executable}")

## パッケージバージョン

In [ ]:
import importlib.metadata

packages = ["ezkl", "torch", "torchvision", "onnx", "numpy", "matplotlib", "pandas"]

for pkg in packages:
    try:
        version = importlib.metadata.version(pkg)
        print(f"  {pkg:<15} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg:<15} ✗ 未インストール")

## torch — GPU / CPU の確認

In [ ]:
import torch

print(f"torch バージョン : {torch.__version__}")
print(f"CUDA 利用可能    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
else:
    print("デバイス         : CPU のみ（Colab GPU ランタイムで再実行すると CUDA が使えます）")

## ezkl の動作確認

In [ ]:
import ezkl
print(f"ezkl バージョン: {ezkl.__version__}")
print(f"利用可能な関数（抜粋）: gen_settings, calibrate_settings, compile_circuit, gen_witness, setup, prove, verify")

## onnx の動作確認

In [ ]:
import onnx
print(f"onnx バージョン: {onnx.__version__}")

## 簡単な torch 動作テスト

In [ ]:
import torch
import torch.nn as nn

x = torch.randn(1, 1, 28, 28)
conv = nn.Conv2d(1, 2, kernel_size=5, stride=2)
out = conv(x)
print(f"入力 shape: {x.shape}")
print(f"Conv2d 出力 shape: {out.shape}  ← (batch=1, ch=2, 12, 12) なら OK")

## matplotlib の動作確認

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.use('Agg')  # GUI なし環境でも動くバックエンド

x = np.linspace(0, 2 * np.pi, 100)
plt.figure(figsize=(6, 3))
plt.plot(x, np.sin(x))
plt.title("matplotlib 動作確認")
plt.tight_layout()
plt.savefig("check_matplotlib.png")
plt.close()
print("check_matplotlib.png を保存しました")

## pandas の動作確認

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "step": ["gen_settings", "calibrate", "compile", "gen_witness", "setup", "prove", "verify"],
    "入力ファイル": ["network.onnx", "network.onnx", "network.onnx", "input.json", "network.compiled", "witness.json", "proof.json"],
    "出力ファイル": ["settings.json", "settings.json(更新)", "network.compiled", "witness.json", "test.pk / test.vk", "proof.json", "True / False"],
})
print(df.to_string(index=False))

---

すべてのセルが正常に実行できれば環境構築完了です。  
次: `simple_demo.ipynb` で EZKL のフルフローを実行してみましょう。